# Optimizers and LR Schedules

| Part | Question |
|---|---|
| 1 | Reproduce Adam by hand (one weight, five gradients) and check every quantity against PyTorch |
| 2 | Turn off bias correction, plot 20 steps both ways, find when the difference stops mattering |
| 3 | Log the update-to-weight ratio for every layer; find the step where warmup stops changing it |
| 4 | Cosine vs WSD, both scheduled for 300 steps and stopped at step 200: which model to keep? |
| 5 | LR sweep at widths 256 / 512 / 1024, mark the minima, choose an LR for width 4096 |
| 6 | Fair comparison: every arm gets its own LR sweep, and the Part 4 result is repeated over 3 seeds |

**How to run:** Runtime → Change runtime type → **T4 GPU**, then Run all. A full run takes about 15–25 minutes.
Set `QUICK = True` for a 2-minute test with tiny settings.

**Outputs:** the first cell mounts Google Drive (approve the pop-up). Every figure (PNG), table (CSV) and the results summary (TXT) is saved to
`My Drive/TheSchoolofAI/optimizers-lr-schedules/outputs/`.

The written conclusions quote the numbers from my full T4 run. A rerun can differ slightly in the last decimals, because GPU kernels are not bit-exact.

In [ ]:
QUICK = False   # True = fast test run with tiny settings

import math
import os
import time
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from matplotlib.ticker import FuncFormatter

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| torch', torch.__version__)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
RESULTS = {}   # every number worth reporting ends up here

# ---- where outputs go: Google Drive in Colab, ./outputs elsewhere ----
DRIVE_DIR = '/content/drive/MyDrive/TheSchoolofAI/optimizers-lr-schedules/outputs'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = DRIVE_DIR
except ImportError:
    OUT_DIR = 'outputs'
if QUICK:
    OUT_DIR += '_quick'          # test runs never overwrite the real results
os.makedirs(OUT_DIR, exist_ok=True)
print('saving outputs to:', OUT_DIR)


def save_fig(name):
    plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/{name}.png', dpi=150, bbox_inches='tight')
    plt.show()


def save_table(df, name):
    df.to_csv(f'{OUT_DIR}/{name}.csv')


def show(df, fmt='{:.4f}'):
    display(df.style.format(fmt, na_rep='—'))


def smooth(values, k=10):
    # trailing k-step mean; returns (x, y) so the curve stays aligned with the real step numbers
    a = np.asarray(values, float)
    if len(a) < k:
        return np.arange(len(a)), a
    return np.arange(k - 1, len(a)), np.convolve(a, np.ones(k) / k, mode='valid')


lr_ticks = FuncFormatter(lambda x, _: f'{x:.0e}'.replace('e-0', 'e-'))

---
## Part 1: Adam by hand

Adam keeps two running averages per weight and corrects them for starting at zero:

$$m_t = \beta_1 m_{t-1} + (1-\beta_1)\,g_t \qquad v_t = \beta_2 v_{t-1} + (1-\beta_2)\,g_t^2$$
$$\hat m_t = \frac{m_t}{1-\beta_1^t} \qquad \hat v_t = \frac{v_t}{1-\beta_2^t} \qquad w_t = w_{t-1} - \eta\,\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}$$

Setup: $w_0 = 1.0$, $\eta = 0.1$, $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\epsilon = 10^{-8}$, gradients $g = [0.5,\,-0.3,\,0.8,\,0.1,\,-0.6]$.

**Step 1 worked by hand**

- $m_1 = 0.9\cdot 0 + 0.1\cdot 0.5 = 0.05$
- $v_1 = 0.999\cdot 0 + 0.001\cdot 0.25 = 0.00025$
- $\hat m_1 = 0.05 / (1-0.9) = 0.5$
- $\hat v_1 = 0.00025 / (1-0.999) = 0.25$
- step $= 0.1 \cdot 0.5 / (\sqrt{0.25} + 10^{-8}) = 0.1$ (to 8 d.p.), so $w_1 = 0.9$

With bias correction the first step is always $\eta\cdot\text{sign}(g_1)$, whatever the size of the gradient.

The code repeats this arithmetic for all five steps in plain Python, then runs `torch.optim.Adam` on the same inputs in float64.
PyTorch stores $m$ and $v$ (as `exp_avg` and `exp_avg_sq`) and applies the parameter update, so those are checked directly against the optimizer.
PyTorch never stores $\hat m$ and $\hat v$, so their PyTorch column is computed from PyTorch's own $m$ and $v$; the step and new weight confirm the full update.

In [ ]:
w0, lr, b1, b2, eps = 1.0, 0.1, 0.9, 0.999, 1e-8
grads = [0.5, -0.3, 0.8, 0.1, -0.6]

# ---- by hand (plain Python floats) ----
w, m, v, rows = w0, 0.0, 0.0, []
for t, g in enumerate(grads, start=1):
    m = b1 * m + (1 - b1) * g
    v = b2 * v + (1 - b2) * g * g
    m_hat = m / (1 - b1 ** t)
    v_hat = v / (1 - b2 ** t)
    step = lr * m_hat / (math.sqrt(v_hat) + eps)
    w = w - step
    rows.append(dict(t=t, g=g, m=m, v=v, m_hat=m_hat, v_hat=v_hat, step=step, w=w))
hand = pd.DataFrame(rows).set_index('t')

# ---- PyTorch, float64 so any mismatch is a real difference, not rounding ----
p = torch.tensor([w0], dtype=torch.float64, requires_grad=True)
opt = torch.optim.Adam([p], lr=lr, betas=(b1, b2), eps=eps)
rows = []
for t, g in enumerate(grads, start=1):
    w_before = p.item()
    p.grad = torch.tensor([g], dtype=torch.float64)
    opt.step()
    m_t = opt.state[p]['exp_avg'].item()
    v_t = opt.state[p]['exp_avg_sq'].item()
    rows.append(dict(t=t, g=g, m=m_t, v=v_t, m_hat=m_t / (1 - b1 ** t), v_hat=v_t / (1 - b2 ** t),
                     step=w_before - p.item(), w=p.item()))
torch_df = pd.DataFrame(rows).set_index('t')

print('By hand:')
show(hand, '{:.10f}')
print('PyTorch:')
show(torch_df, '{:.10f}')

diff = (hand - torch_df).abs()
print('Largest |hand - PyTorch| per column:')
show(diff.max().to_frame('max abs diff'), '{:.2e}')
max_err = diff.values.max()
print(f'Largest disagreement anywhere: {max_err:.2e}  ->  agree to about {int(-math.log10(max_err + 1e-300))} decimal places')

RESULTS['p1_max_abs_diff'] = max_err
save_table(hand, 'p1_adam_by_hand')
save_table(torch_df, 'p1_adam_pytorch')

**Answer.** Every quantity matches PyTorch to about **16 decimal places** (largest difference $3.5\times10^{-17}$, which is float64 rounding).

---
## Part 2: Turning bias correction off

Without correction the update uses the raw $m_t$ and $v_t$. For the same gradient history, the uncorrected step equals the corrected step times

$$r(t) = \frac{1-\beta_1^t}{\sqrt{1-\beta_2^t}}$$

$m$ catches up in about $1/(1-\beta_1) = 10$ steps, while $v$ needs about $1/(1-\beta_2)$ steps.
So which way the uncorrected step is wrong depends on $\beta_2$:

- **$\beta_2 = 0.999$:** $v$ is underestimated far more than $m$, so uncorrected steps are **too large** ($r>1$).
- **$\beta_2 = 0.95$:** $v$ catches up faster than $m$, so uncorrected steps start **too small** ($r(1) = 0.45$) and then overshoot slightly before settling at 1.

Below: 20 steps on a noisy quadratic $f(w)=\tfrac12(w-3)^2$ with both versions, then the exact step after which $r(t)$ stays within 1% and 5% of 1.
The two simulated runs drift apart, so they don't see identical gradients; $r(t)$ is the clean, gradient-independent measure.

In [ ]:
def run_adam(steps, bias_correction, lr=0.1, b1=0.9, b2=0.999, eps=1e-8, seed=0, w0=0.0):
    rng = np.random.default_rng(seed)
    w, m, v = w0, 0.0, 0.0
    ws, step_sizes = [w0], []
    for t in range(1, steps + 1):
        g = (w - 3.0) + rng.normal(0, 0.5)          # noisy gradient of 0.5*(w-3)^2
        m = b1 * m + (1 - b1) * g
        v = b2 * v + (1 - b2) * g * g
        if bias_correction:
            m_used, v_used = m / (1 - b1 ** t), v / (1 - b2 ** t)
        else:
            m_used, v_used = m, v
        s = lr * m_used / (math.sqrt(v_used) + eps)
        w -= s
        ws.append(w)
        step_sizes.append(s)
    return np.array(ws), np.array(step_sizes)


def r_of_t(t, b2, b1=0.9):
    return (1 - b1 ** t) / np.sqrt(1 - b2 ** t)


def steps_until(b2, tol, tmax=200_000):
    # first step after which |r(t) - 1| stays below tol for good
    r = r_of_t(np.arange(1, tmax + 1), b2)
    bad = np.where(np.abs(r - 1) > tol)[0]
    return int(bad[-1] + 2) if len(bad) else 1


T = 20
w_bc, s_bc = run_adam(T, True)
w_nb, s_nb = run_adam(T, False)
t = np.arange(1, T + 1)

fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))
ax[0].plot(range(T + 1), w_bc, 'o-', label='bias-corrected')
ax[0].plot(range(T + 1), w_nb, 's--', label='no bias correction')
ax[0].axhline(3, color='grey', lw=1, ls=':')
ax[0].set(title='Weight trajectory (target w = 3)', xlabel='step', ylabel='w')
ax[0].legend()
ax[1].plot(t, np.abs(s_bc), 'o-', label='bias-corrected')
ax[1].plot(t, np.abs(s_nb), 's--', label='no bias correction')
ax[1].set(title='Step size (lr = 0.1)', xlabel='step', ylabel='|step|')
ax[1].legend()
ax[2].plot(t, r_of_t(t, 0.999), 'o-', label=r'$\beta_2=0.999$')
ax[2].plot(t, r_of_t(t, 0.95), 's-', label=r'$\beta_2=0.95$')
ax[2].axhline(1, color='grey', lw=1, ls=':')
ax[2].set(title='r(t) = uncorrected / corrected step', xlabel='step', ylabel='r(t)')
ax[2].legend()
save_fig('p2_bias_correction')

summary = pd.DataFrame([{
    'beta2': b,
    'r(1)': r_of_t(1, b),
    'r(20)': r_of_t(20, b),
    'stays within 5% from step': steps_until(b, 0.05),
    'stays within 1% from step': steps_until(b, 0.01),
} for b in (0.999, 0.99, 0.95)]).set_index('beta2')
show(summary, '{:.3f}')
save_table(summary, 'p2_bias_correction_steps')

n999, n999_5, n95 = (summary.loc[0.999, 'stays within 1% from step'],
                     summary.loc[0.999, 'stays within 5% from step'],
                     summary.loc[0.95, 'stays within 1% from step'])
RESULTS.update(p2_steps_1pct_b2_0999=n999, p2_steps_5pct_b2_0999=n999_5, p2_steps_1pct_b2_095=n95)
print(f'With beta2 = 0.999 the difference stops mattering after {n999:,} steps (<1%), or {n999_5:,} steps (<5%).')
print(f'With beta2 = 0.95 it stops mattering after {n95} steps (<1%).')

**Answer.** With $\beta_2 = 0.999$ the difference stops mattering (stays within 1%) only after **3,925 steps**, or 2,375 steps at a 5% tolerance.
Over the first 20 steps the uncorrected steps are 3–6× too large: $r(1) = 3.2$, $r(20) = 6.2$.
With the $\beta_2 = 0.95$ common for LLMs it stops mattering after **76 steps**.

LR warmup hides most of this in practice: the LR is tiny during exactly the steps where the uncorrected update is most wrong.

---
## Shared setup for Parts 3–6: a small GPT on Tiny Shakespeare

- Character-level, 2 transformer layers, head size 64, sequence length 128, batch 32.
- AdamW with $\beta=(0.9, 0.95)$, no weight decay, gradient clipping at 1.0.
- Runs with the same seed start from **identical weights and see identical batches**, so differences come from the setting under test only.
- Validation loss is measured on a fixed set of 20 batches.

In [ ]:
if os.path.exists('input.txt'):
    text = open('input.txt').read()
else:
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    text = urllib.request.urlopen(url, timeout=60).read().decode()
    open('input.txt', 'w').write(text)

chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
VOCAB = len(chars)
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
split = int(0.9 * len(data))
train_data, val_data = data[:split], data[split:]

SEQ, BATCH, N_LAYER = (64, 16, 1) if QUICK else (128, 32, 2)
EVAL_BATCHES = 4 if QUICK else 20
print(f'{len(text):,} characters, vocab {VOCAB} | seq {SEQ}, batch {BATCH}, layers {N_LAYER}')


def get_batch(source, gen):
    ix = torch.randint(len(source) - SEQ - 1, (BATCH,), generator=gen)
    x = torch.stack([source[i:i + SEQ] for i in ix])
    y = torch.stack([source[i + 1:i + SEQ + 1] for i in ix])
    return x.to(device), y.to(device)


val_gen = torch.Generator().manual_seed(1234)
VAL_BATCHES = [get_batch(val_data, val_gen) for _ in range(EVAL_BATCHES)]


class CausalSelfAttention(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.n_head = max(1, d // 64)
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = [z.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) for z in (q, k, v)]
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, C))


class Block(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.attn = CausalSelfAttention(d)
        self.ln2 = nn.LayerNorm(d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))


class GPT(nn.Module):
    def __init__(self, d, n_layer=N_LAYER):
        super().__init__()
        self.tok = nn.Embedding(VOCAB, d)
        self.pos = nn.Embedding(SEQ, d)
        self.blocks = nn.Sequential(*[Block(d) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, VOCAB, bias=False)

    def forward(self, x, y=None):
        h = self.tok(x) + self.pos(torch.arange(x.shape[1], device=x.device))
        logits = self.head(self.ln_f(self.blocks(h)))
        if y is None:
            return logits
        return F.cross_entropy(logits.view(-1, VOCAB), y.view(-1))


def lr_at(step, total, peak, kind, warmup, min_frac=0.1, wsd_decay_frac=0.2):
    if warmup and step < warmup:
        return peak * (step + 1) / warmup
    if kind == 'const':
        return peak
    if kind == 'cosine':
        progress = min((step - warmup) / max(1, total - warmup), 1.0)
        return peak * (min_frac + (1 - min_frac) * 0.5 * (1 + math.cos(math.pi * progress)))
    if kind == 'wsd':
        decay_start = int(total * (1 - wsd_decay_frac))
        if step < decay_start:
            return peak
        progress = min((step - decay_start) / max(1, total - decay_start), 1.0)
        return peak * (1 - (1 - min_frac) * progress)
    raise ValueError(kind)


@torch.no_grad()
def evaluate(model):
    model.eval()
    loss = np.mean([model(x, y).item() for x, y in VAL_BATCHES])
    model.train()
    return loss if math.isfinite(loss) else float('nan')


def train_step(model, opt, gen, lr):
    for group in opt.param_groups:
        group['lr'] = lr
    x, y = get_batch(train_data, gen)
    loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    return loss


def train(width, peak_lr, total, kind='cosine', warmup=30, stop_at=None, seed=0, log_ratio=False):
    # seed fixes both the initial weights and the order of training batches
    torch.manual_seed(seed)
    model = GPT(width).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=peak_lr, betas=(0.9, 0.95), weight_decay=0.0)
    gen = torch.Generator().manual_seed(seed)
    matrices = {n: p for n, p in model.named_parameters() if p.dim() == 2}
    hist = dict(loss=[], lr=[], ratio={n: [] for n in matrices})

    for step in range(stop_at or total):
        lr = lr_at(step, total, peak_lr, kind, warmup)
        loss = train_step(model, opt, gen, lr)
        if log_ratio:
            before = {n: p.detach().clone() for n, p in matrices.items()}
        opt.step()
        if log_ratio:
            for n, p in matrices.items():
                hist['ratio'][n].append(((p.detach() - before[n]).norm() / before[n].norm()).item())
        hist['loss'].append(loss.item())
        hist['lr'].append(lr)
        if not math.isfinite(loss.item()):
            break        # diverged: stop and report NaN
    return dict(val=evaluate(model), hist=hist, model=model, opt=opt, gen=gen)


def cooldown(run, peak_lr, steps, min_frac=0.1):
    # continue a stopped run with a short linear decay from peak_lr to min_frac * peak_lr
    losses = []
    for i in range(steps):
        lr = peak_lr * (1 - (1 - min_frac) * (i + 1) / steps)
        loss = train_step(run['model'], run['opt'], run['gen'], lr)
        run['opt'].step()
        losses.append(loss.item())
    return evaluate(run['model']), losses


t0 = time.time()
sanity = train(256, 3e-3, 20)
print(f'sanity run: val loss {sanity["val"]:.3f} after 20 steps ({time.time() - t0:.1f}s)')

---
## Part 3: Update-to-weight ratio per layer, and when warmup stops mattering

For every weight matrix (embeddings, attention and MLP projections, output head) we log $\;\lVert W_{t+1}-W_t\rVert \,/\, \lVert W_t\rVert\;$ at every step.
Biases and LayerNorm gains are left out: they are vectors that start near 0 or 1, so their ratio says little about the layer's learning speed.

To isolate warmup, we train **twice from the identical start**: once with 100 steps of linear warmup and once with none, both at the same constant peak LR afterwards.
For each layer we report two steps:

- **Merge step:** the first step after which the warmed-up and cold-start ratios stay within 10% of each other (both as 10-step means). This is when warmup stops changing the ratio.
- **Plateau step:** the first step after which the warmed-up ratio stays within 10% of its own final level (mean of the last 50 steps).

In [ ]:
P3_LR = 3e-3
P3_STEPS, P3_WARM, P3_WIDTH = (60, 20, 128) if QUICK else (300, 100, 256)
warm = train(P3_WIDTH, P3_LR, P3_STEPS, kind='const', warmup=P3_WARM, log_ratio=True)
cold = train(P3_WIDTH, P3_LR, P3_STEPS, kind='const', warmup=0, log_ratio=True)


def last_violation(rel, tol, x):
    bad = np.where(rel > tol)[0]
    if len(bad) == 0:
        return int(x[0])
    if bad[-1] == len(rel) - 1:
        return None             # still outside the band at the end of the run
    return int(x[bad[-1] + 1])


def merge_step(a, b, tol=0.10):
    x, sa = smooth(a)
    _, sb = smooth(b)
    return last_violation(np.abs(sa - sb) / np.maximum(np.abs(sb), 1e-12), tol, x)


def plateau_step(a, tol=0.10, tail=50):
    x, sa = smooth(a)
    final = np.mean(a[-tail:])
    return last_violation(np.abs(sa - final) / final, tol, x)


rows = []
for name in warm['hist']['ratio']:
    a, b = warm['hist']['ratio'][name], cold['hist']['ratio'][name]
    rows.append({'layer': name,
                 'merge step': merge_step(a, b),
                 'plateau step': plateau_step(a),
                 'final ratio (warmup)': np.mean(a[-20:]),
                 'final ratio (no warmup)': np.mean(b[-20:])})
p3 = pd.DataFrame(rows).set_index('layer')
display(p3.style.format({'merge step': lambda v: 'not within run' if pd.isna(v) else f'{int(v)}',
                         'plateau step': lambda v: 'not within run' if pd.isna(v) else f'{int(v)}',
                         'final ratio (warmup)': '{:.2e}', 'final ratio (no warmup)': '{:.2e}'}))
save_table(p3, 'p3_merge_step_per_layer')

merges = p3['merge step'].dropna().astype(int).tolist()
plateaus = p3['plateau step'].dropna().astype(int).tolist()
RESULTS['p3_warmup_steps'] = P3_WARM
if merges:
    RESULTS['p3_median_merge'] = int(np.median(merges))
    RESULTS['p3_merge_range'] = (min(merges), max(merges))
    print(f'Warmup ends at step {P3_WARM}. Warmup stops changing the ratio at step ~{int(np.median(merges))} '
          f'(median over layers; range {min(merges)}–{max(merges)}).')
if plateaus:
    RESULTS['p3_median_plateau'] = int(np.median(plateaus))
    print(f'The warmed-up ratio reaches its final level at step ~{int(np.median(plateaus))} (median over layers).')
print(f'Final val loss: warmup {warm["val"]:.4f}, no warmup {cold["val"]:.4f}')

names = list(warm['hist']['ratio'])
shown = [n for n in names if any(k in n for k in ('tok', 'blocks.0.attn.qkv', 'blocks.0.mlp.0',
                                                  f'blocks.{N_LAYER - 1}.mlp.2', 'head'))]
fig, ax = plt.subplots(1, 2, figsize=(15, 4.6), sharey=True)
for a, run, title in ((ax[0], warm, f'with {P3_WARM}-step warmup'), (ax[1], cold, 'no warmup')):
    for n in shown:
        a.plot(*smooth(run['hist']['ratio'][n]), label=n)
    a.axvline(P3_WARM, color='grey', ls=':', lw=1)
    a.set_yscale('log')
    a.set(title=f'Update-to-weight ratio, {title}', xlabel='step')
ax[0].set_ylabel('‖ΔW‖ / ‖W‖  (10-step mean)')
ax[1].legend(fontsize=8)
save_fig('p3_update_ratio_per_layer')

fig, ax = plt.subplots(figsize=(9, 4))
for n in shown:
    x, sa = smooth(warm['hist']['ratio'][n])
    _, sb = smooth(cold['hist']['ratio'][n])
    ax.plot(x, sa / sb, label=n)
ax.axhline(1, color='grey', lw=1)
ax.axhspan(0.9, 1.1, color='grey', alpha=0.15, label='±10% band')
ax.axvline(P3_WARM, color='grey', ls=':')
ax.set(title='Warmup run ÷ no-warmup run: when the curves enter the band, warmup has stopped mattering',
       xlabel='step', ylabel='ratio (warmup) / ratio (no warmup)')
ax.legend(fontsize=7)
save_fig('p3_warmup_vs_nowarmup_ratio')

**Answer.** Warmup ends at step 100, but it keeps changing the update-to-weight ratio until about **step 177**
(median over the 8 weight matrices that converge; range 109–287).
The effect of warmup lasts most of another warmup-length after the warmup itself.
The warmed-up ratios level off at about step 219 (median plateau step).

Three MLP matrices never converge within 300 steps: both MLP layers in block 0 and the output MLP layer in block 1.
At step 300 their update-to-weight ratio is still 13–25% higher in the warmed-up run.

Why: during warmup the ratio rises roughly in step with the LR, because Adam's update size is about LR × a normalised direction.
But warmup also changes how the weight norms grow in the early steps, so the two runs travel different paths,
and the ratio only matches once their weight norms have converged.
For the MLP layers that never converge, warmup leaves a lasting mark on where the weights end up, not just on the first few updates.

Caveat: this compares one pair of runs. Rerunning the notebook moved the median from 204 to 177, so treat these steps as roughly ±30.

---
## Part 4: Cosine vs WSD, scheduled for 300 steps, stopped at step 200

- **Cosine:** 30-step warmup, then cosine decay to 10% of peak at step 300.
- **WSD (warmup-stable-decay):** 30-step warmup, constant until step 240, then linear decay to 10% of peak at step 300.

Both are stopped at step 200. **Each schedule gets its own 5-point LR sweep**, and each is compared at its own best LR.
Two extra checks help decide which model to keep:

- **Cooldown:** the WSD checkpoint at step 200 gets a 30-step linear decay, which is how a WSD run is meant to finish early.
- **Full run:** what each schedule reaches if allowed to run all 300 steps.

In [ ]:
P4_WIDTH = 128 if QUICK else 256
P4_TOTAL, P4_STOP, P4_WARM = (60, 40, 6) if QUICK else (300, 200, 30)
P4_LRS = [1e-3, 2e-3, 4e-3, 8e-3] if QUICK else [1e-3, 2e-3, 4e-3, 8e-3, 1.6e-2]
COOL = 6 if QUICK else 30

sweep4 = {}
for kind in ('cosine', 'wsd'):
    for lr in P4_LRS:
        sweep4[(kind, lr)] = train(P4_WIDTH, lr, P4_TOTAL, kind=kind, warmup=P4_WARM, stop_at=P4_STOP)['val']
        print(f'{kind:6s} lr={lr:.1e}  val loss @ step {P4_STOP} = {sweep4[(kind, lr)]:.4f}')
tab4 = pd.Series(sweep4).unstack(0)
show(tab4)
save_table(tab4, 'p4_lr_sweep_val_at_stop')

best = {}
for kind in ('cosine', 'wsd'):
    best[kind] = tab4[kind].idxmin()
    on_edge = best[kind] in (P4_LRS[0], P4_LRS[-1])
    print(f'best LR for {kind}: {best[kind]:.0e}' + ('   <- ON GRID EDGE: widen the sweep before trusting this' if on_edge else ''))

cos_run = train(P4_WIDTH, best['cosine'], P4_TOTAL, 'cosine', P4_WARM, stop_at=P4_STOP)
wsd_run = train(P4_WIDTH, best['wsd'], P4_TOTAL, 'wsd', P4_WARM, stop_at=P4_STOP)
wsd_lr_at_stop = wsd_run['hist']['lr'][-1]
wsd_cool_val, cool_losses = cooldown(wsd_run, wsd_lr_at_stop, COOL)
cos_full = train(P4_WIDTH, best['cosine'], P4_TOTAL, 'cosine', P4_WARM)['val']
wsd_full = train(P4_WIDTH, best['wsd'], P4_TOTAL, 'wsd', P4_WARM)['val']

res4 = pd.DataFrame({
    f'val loss @ {P4_STOP}': [cos_run['val'], wsd_run['val']],
    f'LR @ {P4_STOP}': [cos_run['hist']['lr'][-1], wsd_lr_at_stop],
    f'@ {P4_STOP} + {COOL}-step cooldown': [np.nan, wsd_cool_val],
    f'run to {P4_TOTAL}': [cos_full, wsd_full],
}, index=[f'cosine (lr {best["cosine"]:.0e})', f'WSD (lr {best["wsd"]:.0e})'])
show(res4)
save_table(res4, 'p4_cosine_vs_wsd')
RESULTS['p4_table'] = res4

fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))
steps = np.arange(P4_TOTAL)
ax[0].plot(steps, [lr_at(s, P4_TOTAL, 1, 'cosine', P4_WARM) for s in steps], label='cosine')
ax[0].plot(steps, [lr_at(s, P4_TOTAL, 1, 'wsd', P4_WARM) for s in steps], label='WSD')
ax[0].axvline(P4_STOP, color='red', ls='--', label=f'stop @ {P4_STOP}')
ax[0].set(title='Schedules', xlabel='step', ylabel='LR (fraction of peak)')
ax[0].legend()
ax[1].plot(*smooth(cos_run['hist']['loss']), label='cosine')
ax[1].plot(*smooth(wsd_run['hist']['loss']), label='WSD')
ax[1].plot(np.arange(P4_STOP, P4_STOP + COOL), cool_losses, color='C1', ls=':', label='WSD cooldown (per step)')
ax[1].set(title='Training loss', xlabel='step', ylabel='train loss (10-step mean)')
ax[1].legend()
for kind, colour in (('cosine', 'C0'), ('wsd', 'C1')):
    ax[2].plot(tab4.index, tab4[kind], 'o-', color=colour, label=kind)
ax[2].set_xscale('log')
ax[2].set_xticks(P4_LRS)
ax[2].xaxis.set_major_formatter(lr_ticks)
ax[2].minorticks_off()
ax[2].set(title=f'LR sweep (each schedule tuned)', xlabel='peak LR', ylabel=f'val loss @ step {P4_STOP}')
ax[2].legend()
save_fig('p4_cosine_vs_wsd')

**Answer.** Both schedules picked LR 4e-3, which is inside the sweep range, so both are properly tuned.

| | Val loss @ 200 | LR @ 200 | + 30-step cooldown | Run to 300 |
|---|---|---|---|---|
| Cosine | 2.0244 | 1.5e-3 | — | 1.9322 |
| WSD | 2.0294 | 4.0e-3 | 1.9438 | **1.8740** |

At step 200 the two models are tied. The 0.005 gap is smaller than the seed-to-seed spread in Part 6,
and across 3 seeds WSD is actually ahead at step 200 (2.009 vs 2.019, still a tie).

**I would keep the WSD model**, for three reasons:

1. It reaches the same loss while still at full LR. Cosine has already cut its LR to about 37% of peak to get there.
2. The WSD checkpoint can still be trained further or finished with a short cooldown: 30 decay steps take it to 1.944,
   within 0.012 of what cosine reaches after all 300 steps (WSD at 230 steps vs cosine at 300).
   The cosine checkpoint is locked into a schedule that assumed 300 steps.
3. When both run the full 300 steps, WSD finishes 0.058 lower, and Part 6 shows that lead holds on all 3 seeds.

If no more compute were available at all, the two step-200 models are interchangeable, so the choice would not matter.

---
## Part 5: LR sweep across widths, extrapolated to 4096

Same model, data and 300-step cosine schedule, using standard PyTorch parameterisation (not μP).
For each width we sweep 9 log-spaced LRs and plot validation loss against LR.

The minimum is estimated by fitting a parabola in $\log_{10}(\text{LR})$ through the best grid point and its two neighbours.
The fit is only used if it is a real valley (opens upward) with its lowest point between the neighbours; otherwise the best grid point is used.
We then fit $\log \text{LR}^* = a + b\,\log(\text{width})$ and extrapolate to width 4096.

In [ ]:
WIDTHS = [64, 128, 256] if QUICK else [256, 512, 1024]
TARGET = 1024 if QUICK else 4096
P5_STEPS, P5_WARM = (40, 4) if QUICK else (300, 30)
LR_GRID = np.logspace(-3.5, -1.5, 5 if QUICK else 9)

sweep5 = {}
for width in WIDTHS:
    t0 = time.time()
    for lr in LR_GRID:
        sweep5[(width, lr)] = train(width, lr, P5_STEPS, 'cosine', P5_WARM)['val']
    print(f'width {width}: done in {time.time() - t0:.0f}s')
tab5 = pd.Series(sweep5).unstack(0)
show(tab5)
save_table(tab5, 'p5_lr_sweep_by_width')


def fitted_minimum(lrs, losses):
    x = np.log10(lrs)
    y = np.asarray(losses, float)
    y = np.where(np.isfinite(y), y, np.nanmax(y[np.isfinite(y)]) + 1)   # diverged runs count as worst
    i = int(np.argmin(y))
    if i == 0 or i == len(x) - 1:
        return 10 ** x[i], y[i], 'grid edge'
    a, b, c = np.polyfit(x[i - 1:i + 2], y[i - 1:i + 2], 2)
    xm = -b / (2 * a) if a > 0 else None
    if xm is None or not (x[i - 1] <= xm <= x[i + 1]):
        return 10 ** x[i], y[i], 'grid point'
    return 10 ** xm, np.polyval([a, b, c], xm), 'parabola'


minima = {w: fitted_minimum(tab5.index.values, tab5[w].values) for w in WIDTHS}
for w, (lr_star, loss_star, how) in minima.items():
    warning = '   <- ON GRID EDGE: widen the sweep' if how == 'grid edge' else ''
    print(f'width {w:5d}: best LR = {lr_star:.2e}  (val loss {loss_star:.4f}, from {how}){warning}')

fig, ax = plt.subplots(figsize=(8.5, 5.2))
for i, w in enumerate(WIDTHS):
    ax.plot(tab5.index, tab5[w], 'o-', color=f'C{i}', label=f'width {w}')
    lr_star, loss_star, _ = minima[w]
    ax.plot(lr_star, loss_star, '*', color=f'C{i}', ms=18, mec='k', zorder=5)
    ax.annotate(f'{lr_star:.1e}', (lr_star, loss_star), textcoords='offset points', xytext=(0, -22),
                ha='center', fontsize=9, color=f'C{i}', fontweight='bold')
finite = tab5.values[np.isfinite(tab5.values)]
ax.set_ylim(finite.min() - 0.12, min(finite.max(), np.percentile(finite, 80) + 0.4))
ax.set_xscale('log')
ax.xaxis.set_major_formatter(lr_ticks)
ax.set(xlabel='peak learning rate', ylabel=f'val loss after {P5_STEPS} steps',
       title='LR sweep across widths (★ = estimated minimum)')
ax.legend()
save_fig('p5_lr_sweep_by_width')

log_w = np.log2(WIDTHS)
log_lr = np.log2([minima[w][0] for w in WIDTHS])
slope, intercept = np.polyfit(log_w, log_lr, 1)
pred = 2 ** (intercept + slope * np.log2(TARGET))
pair_preds = [2 ** (log_lr[j] + (log_lr[k] - log_lr[j]) / (log_w[k] - log_w[j]) * (np.log2(TARGET) - log_w[j]))
              for j in range(len(WIDTHS)) for k in range(j + 1, len(WIDTHS))]

print(f'\nfit: best LR ∝ width^{slope:.2f}   (standard parameterisation predicts about width^-1; μP would give about width^0)')
print(f'prediction for width {TARGET}: {pred:.2e}')
print(f'range from extrapolating each pair of widths: {min(pair_preds):.1e} – {max(pair_preds):.1e}')
RESULTS.update(p5_minima={w: minima[w][0] for w in WIDTHS}, p5_exponent=slope, p5_pred=pred,
               p5_pair_range=(min(pair_preds), max(pair_preds)), p5_target=TARGET)

fig, ax = plt.subplots(figsize=(6.5, 4.4))
ax.loglog(WIDTHS, [minima[w][0] for w in WIDTHS], 'o', ms=9, label='measured best LR')
xs = np.array([WIDTHS[0], TARGET])
ax.loglog(xs, 2 ** (intercept + slope * np.log2(xs)), '--', label=f'fit: ∝ width^{slope:.2f}')
ax.errorbar([TARGET], [pred], yerr=[[pred - min(pair_preds)], [max(pair_preds) - pred]],
            fmt='*', ms=16, mec='k', color='C2', capsize=6, label=f'width {TARGET}: {pred:.1e} (pairwise range)')
ax.set(xlabel='width', ylabel='best LR', title='Extrapolating the best LR')
ax.legend(fontsize=8)
save_fig('p5_lr_extrapolation')

**Answer.**

| Width | 256 | 512 | 1024 |
|---|---|---|---|
| Best LR | 5.76e-3 | 2.63e-3 | 1.38e-3 |

The best LR falls as **width^−1.03**, almost exactly the width^−1 that standard parameterisation predicts for Adam.
Each doubling of width roughly halves the best LR.

**At width 4096 I would use about 3e-4.** The fitted line gives 3.2e-4, and extrapolating from each pair of widths gives 2.5e-4 to 3.8e-4.

**Confidence: moderate, within about 1.5× either way.** The three minima line up well and the slope matches theory, but:
- the estimate goes two doublings past the widest model I measured;
- each minimum comes from a single-seed, 300-step sweep, and a rerun shifted the width-256 minimum from 6.5e-3 to 5.8e-3;
- one LR is shared by embeddings and hidden layers, which do not scale the same way with width.

Before a long run at width 4096 I would check 2e-4, 3e-4 and 4.5e-4.
With μP the best LR would stay roughly constant across widths, and this extrapolation would not be needed.

---
## Part 6: Tune both sides before accepting a comparison

How this notebook follows that rule:

- **Part 4:** cosine and WSD each get their own LR sweep, are compared at their own best LR, and a best LR on the edge of the sweep is flagged.
- **Part 5:** every width gets the full LR grid, and a minimum on the grid edge is flagged as not properly tuned.
- **Same seed → same initial weights and same batches** for every arm, so the only difference is the setting under test.

A tuned comparison can still be fooled by luck, so the last check repeats the Part 4 comparison with 3 seeds.
Each seed changes both the initial weights and the batch order. A gap between schedules only counts if it is larger than the spread across seeds.

In [ ]:
SEEDS = [0, 1] if QUICK else [0, 1, 2]
rows = []
for kind in ('cosine', 'wsd'):
    for seed in SEEDS:
        stopped = train(P4_WIDTH, best[kind], P4_TOTAL, kind, P4_WARM, stop_at=P4_STOP, seed=seed)
        row = {'schedule': kind, 'seed': seed, f'val @ {P4_STOP}': stopped['val']}
        row['+ cooldown'] = cooldown(stopped, stopped['hist']['lr'][-1], COOL)[0] if kind == 'wsd' else np.nan
        row[f'run to {P4_TOTAL}'] = train(P4_WIDTH, best[kind], P4_TOTAL, kind, P4_WARM, seed=seed)['val']
        rows.append(row)
        print(f'{kind:6s} seed {seed}: val @ {P4_STOP} = {row[f"val @ {P4_STOP}"]:.4f}, run to {P4_TOTAL} = {row[f"run to {P4_TOTAL}"]:.4f}')

seeds_df = pd.DataFrame(rows).set_index(['schedule', 'seed'])
save_table(seeds_df, 'p6_seed_repeats')
agg = seeds_df.groupby(level='schedule').agg(['mean', 'std'])
show(agg)
save_table(agg, 'p6_seed_summary')

verdicts = []
for col in (f'val @ {P4_STOP}', f'run to {P4_TOTAL}'):
    c, w = seeds_df.loc['cosine', col], seeds_df.loc['wsd', col]
    gap = c.mean() - w.mean()
    noise = math.sqrt(c.std() ** 2 / len(c) + w.std() ** 2 / len(w))   # standard error of the gap
    leader = 'WSD' if gap > 0 else 'cosine'
    verdict = (f'{col}: cosine {c.mean():.4f} ± {c.std():.4f}, WSD {w.mean():.4f} ± {w.std():.4f} '
               f'-> gap {abs(gap):.4f} in favour of {leader}, '
               + ('larger than seed noise' if abs(gap) > 2 * noise else 'within seed noise (a tie)')
               + f' (2 × std. error = {2 * noise:.4f})')
    verdicts.append(verdict)
    print(verdict)
RESULTS['p6_verdicts'] = verdicts

**Answer.** Repeating the Part 4 comparison over 3 seeds (each changes the initial weights and the batch order):

| | Cosine | WSD | Gap | 2 × std. error | Verdict |
|---|---|---|---|---|---|
| Val loss @ 200 | 2.019 ± 0.014 | 2.009 ± 0.016 | 0.010 for WSD | 0.025 | **tie** |
| Run to 300 | 1.925 ± 0.011 | 1.864 ± 0.010 | 0.061 for WSD | 0.017 | **WSD better** |

- **At step 200 the schedules are tied.** Cosine won on seed 0 and WSD on seeds 1 and 2, and the gap is well within seed noise.
  A single run would have picked whichever schedule that seed happened to favour; my two seed-0 runs alone differed by up to 0.005.
- **At step 300 WSD is genuinely better.** It wins on every seed, and the gap is more than 3× the noise level.
- WSD with a 30-step cooldown from step 200 averages 1.939, within 0.015 of cosine's full 300-step result.

So the Part 4 decision holds up: the step-200 models are equal, and the WSD run is the better one to keep.
Because both schedules were tuned separately, the step-300 gap reflects the schedule itself rather than one side getting a better LR.

In [ ]:
lines = ['=================== RESULTS SUMMARY ===================',
         f"P1  hand vs PyTorch, largest difference: {RESULTS['p1_max_abs_diff']:.2e}",
         f"P2  bias correction stops mattering (<1%): beta2=0.999 after {RESULTS['p2_steps_1pct_b2_0999']:,} steps "
         f"(<5%: {RESULTS['p2_steps_5pct_b2_0999']:,}); beta2=0.95 after {RESULTS['p2_steps_1pct_b2_095']} steps"]
if 'p3_median_merge' in RESULTS:
    lines.append(f"P3  warmup ({RESULTS['p3_warmup_steps']} steps) stops changing the update/weight ratio at step "
                 f"~{RESULTS['p3_median_merge']} (layers {RESULTS['p3_merge_range'][0]}–{RESULTS['p3_merge_range'][1]})")
lines += ['P4', RESULTS['p4_table'].to_string(float_format=lambda v: f'{v:.4f}')]
lines.append('P5  best LR: ' + ', '.join(f'width {w}: {v:.2e}' for w, v in RESULTS['p5_minima'].items())
             + f" | best LR ∝ width^{RESULTS['p5_exponent']:.2f} | width {RESULTS['p5_target']}: {RESULTS['p5_pred']:.2e} "
             f"(range {RESULTS['p5_pair_range'][0]:.1e}–{RESULTS['p5_pair_range'][1]:.1e})")
lines += ['P6  ' + v for v in RESULTS['p6_verdicts']]
report = '\n'.join(lines)
print(report)
with open(f'{OUT_DIR}/results_summary.txt', 'w') as f:
    f.write(report + '\n')
print('\nSaved to', OUT_DIR, ':')
print(sorted(os.listdir(OUT_DIR)))